# Bölüm 6: NumPy & PyTorch Hayatta Kalma Rehberi
**İlk LLM'inizi Oluşturun — Bölüm 6: NumPy & PyTorch Hayatta Kalma Rehberi**

Tensör temelleri, maskeleme, yayma ve küçük bir eğitim döngüsü görmek için bu hücreleri yukarıdan aşağıya çalıştırın.

In [ ]:
# ===== İÇE AKTARMALAR =====
# torch: Tensör işlemleri için PyTorch kütüphanesi (düşünün: çok boyutlu diziler)
# numpy: Klasik sayısal hesaplama kütüphanesi (PyTorch ondan ilham alır)
# torch.nn: Sinir ağı yapı taşları (katmanlar, vb.)
import torch, numpy as np
import torch.nn as nn

print('Torch versiyonu:', torch.__version__)
print('CUDA mevcut:', torch.cuda.is_available())
# CUDA = GPU hesaplama. True ise GPU hızlandırma kullanabiliriz (derin öğrenme için 10-100× daha hızlı)

## Tensör nedir?

Bir **tensör** çok boyutlu bir sayı dizisidir:
- **1D tensör** = bir liste `[1, 2, 3]` (elektronik tabloda bir satır gibi)
- **2D tensör** = bir tablo/matris (satırlar ve sütunlar)
- **3D tensör** = tabloların yığını (Excel'de birden fazla sayfa gibi)

**Neden NumPy yerine PyTorch?**
Her ikisi de çok boyutlu dizileri işler, ancak PyTorch ekler:
1. **GPU desteği** — 10-100× hızlanma için veriyi GPU'ya taşı
2. **Otomatik gradyanlar** — sinir ağlarını eğitmek için türev hesapla
3. **Sinir ağı katmanları** — önceden oluşturulmuş yapı taşları

## Tensör oluşturma
Python/NumPy verisinden ve yaygın doldurma kurallarıyla.

In [ ]:
# ===== Tensör Oluşturma =====

# Python verisinden
data = torch.tensor([[1, 2, 3], [4, 5, 6]])  # 2×3 tensör

# Doldurma kuralları (belirli değerlerle doldurulmuş tensörler oluştur)
zeros = torch.zeros(3, 4)     # 3×4 sıfır tensörü
ones = torch.ones(2, 3, 4)    # 2×3×4 birler tensörü
uniform = torch.rand(3, 4)    # [0, 1) aralığında rastgele değerler
normal = torch.randn(3, 4)    # Normal dağılımdan rastgele değerler (ortalama=0, std=1)
integers = torch.randint(0, 10, (3, 4))  # [0, 10) aralığında rastgele tamsayılar

# Aralıklı diziler (Python'un range'i gibi)
sequence = torch.arange(0, 10, 2)  # [0, 2, 4, 6, 8]
linspace = torch.linspace(0, 1, 5)  # 0'dan 1'e 5 eşit aralıklı nokta

# Veri tipleri (dtype) — hassasiyeti kontrol et
float_tensor = torch.tensor([1.0, 2.0], dtype=torch.float32)  # 32-bit float'lar (varsayılan)
int_tensor = torch.tensor([1, 2], dtype=torch.long)           # 64-bit tamsayılar (indeksler için)

# NumPy ↔ PyTorch (belleği paylaşırlar — birindeki değişiklikler diğerini etkiler!)
np_data = np.array([1, 2, 3], dtype=np.float32)
torch_from_np = torch.from_numpy(np_data)  # np_data ile bellek paylaşır
back_to_np = torch_from_np.numpy()         # NumPy'a geri dön

# Cihaz yerleştirme — CPU veya GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_tensor = torch.randn(3, 4, device=device, dtype=torch.float32)

print('zeros şekli:', zeros.shape)
print('gpu_tensor cihazı:', gpu_tensor.device)

## Tekrarlanabilirlik: Tohum Ayarlama
Hata ayıklama ve deneyleri karşılaştırma için kritik

In [ ]:
# Tekrarlanabilirlik için rastgele durumu sabitle
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Bunlar artık her çalıştırmada aynı olacak
a = torch.randn(3, 4)
b = torch.randn(3, 4)

# Tohum olmadan dene - sonuçlar her seferinde değişir
# Ama tohum ile tekrarlanabilirler
torch.manual_seed(42)
x1 = torch.randn(2, 3)
torch.manual_seed(42)
x2 = torch.randn(2, 3)
print('Tohum ile tensörler eşleşir:', torch.allclose(x1, x2))

# Bu neden önemli: Hata ayıklama ve araştırma için tekrarlanabilir deneyler

## Listelerden Tensörlere: Boyutları Oluşturma
Bölüm 5'teki basit listeleri çok boyutlu tensörlere bağla

In [ ]:
# Adım 1: 1D tensörler (Bölüm 5 hatırlatma)
# Bölüm 5'ten: tokenizer'dan token kimlikleri
token_ids = [2, 3, 4, 6]
tokens_1d = torch.tensor(token_ids)
print(f"1D şekil: {tokens_1d.shape}")  # torch.Size([4])
print(f"Veri: {tokens_1d}")

In [ ]:
# Adım 2: Yığınlama (1D → 2D)
# Birden fazla cümleyi aynı anda işle
batch = torch.tensor([
    [2, 3, 4, 6],    # cümle 1
    [5, 7, 8, 9]     # cümle 2
])
print(f"2D yığın şekli: {batch.shape}")  # torch.Size([2, 4])
print("İlk cümle:", batch[0])
print("İlk cümlenin ikinci token'ı:", batch[0, 1])

In [ ]:
# Adım 3: Gömmeler (2D → 3D)
# Her token için 768 sayı ekle (GPT-2 tarzı)
embeddings_3d = torch.randn(2, 4, 768)
print(f"3D gömmeler şekli: {embeddings_3d.shape}")  # torch.Size([2, 4, 768])

# Boyutlarda gezin: yığın → token → özellikler
print("İlk cümle gömmeleri:", embeddings_3d[0].shape)      # (4, 768)
print("İlk cümlenin ilk token'ı:", embeddings_3d[0, 0].shape)  # (768,)

In [ ]:
# Adım 4: Dikkat başları ön izleme (3D → 4D)
# Çok başlı dikkat başka bir boyut ekler (detaylar için endişelenmeyin henüz)
attention_4d = torch.randn(2, 8, 4, 4)  # (yığın, başlar, dizi, dizi)
print(f"4D dikkat şekli: {attention_4d.shape}")
print("Şekil yorumu: (yığın, başlar, dizi_uzunluğu, dizi_uzunluğu)")

## Yeniden şekillendirme, sıkıştırma, permütasyon
Sürekli kullanacağınız şekil jimnastiği.

In [ ]:
# ===== Yeniden Şekillendirme İşlemleri =====
# Yeniden şekillendirme = değerleri değiştirmeden veriyi yeniden düzenlemek (depodaki kutuları yeniden düzenlemek gibi)

x = torch.arange(12)               # 1D tensör oluştur [0,1,2,...,11], şekil (12,)
print(f'Başlangıç: {x.shape}')

# view() — yeniden şekillendir ama "bitişik" bellek gerektirir (veriler sırayla yerleştirilmiş)
x = x.view(3, 4)                   # (3, 4)'e yeniden şekillendir: 3 satır × 4 sütun
print(f'view(3,4) sonrası: {x.shape}')

# reshape() — view() gibi ama bitişik olmayan tensörleri işler (daha güvenli, emin değilseniz bunu kullanın)
x = x.reshape(2, 2, 3)             # (2, 2, 3)'e yeniden şekillendir
print(f'reshape(2,2,3) sonrası: {x.shape}')

# -1 "bu boyutu benim için hesapla" demektir
x = x.view(-1, 3)                  # -1 4 olur (12 toplam eleman ÷ 3 = 4)
print(f'view(-1,3) sonrası: {x.shape}')

# unsqueeze/squeeze — boyut 1'li boyutları ekle veya kaldır
y = torch.randn(3, 4)
y = y.unsqueeze(0)                 # (1, 3, 4) — 0 pozisyonuna yığın boyutu ekle
y = y.unsqueeze(-1)                # (1, 3, 4, 1) — sona boyut ekle
y = y.squeeze()                    # TÜM boyut-1 boyutları kaldır → (3, 4)
print(f'squeeze sonrası: {y.shape}')

# permute — boyutları yeniden sırala (transpoz gibi ama herhangi bir sayıda boyut için)
z = torch.randn(2, 3, 4)           # (yığın, dizi, özellikler)
z = z.permute(0, 2, 1)             # (yığın, özellikler, dizi) — son iki boyutu değiştir
print(f'permute sonrası: {z.shape}')

# flatten — boyutları daralt
t = torch.randn(2, 3, 4)
t_flat = t.flatten()               # (24,) — tüm boyutlar daraltıldı
t_flat_features = t.flatten(1)     # (2, 12) — boyut 1'den başlayarak daralt
print(f'Tamamen düzleştirilmiş: {t_flat.shape}')
print(f'Özellikleri düzleştir: {t_flat_features.shape}')

## İndeksleme ve dilimleme
Yığınları, token'ları seç ve boolean maskeleri kullan.

In [ ]:
# ===== İndeksleme ve Dilimleme =====
# Çok boyutlu veride gezinme klasörlerde gezinir gibidir: yığın → token → özellikler

# Dikkati simüle eden 4D tensör oluştur: (yığın, başlar, dizi, dizi)
x = torch.randn(2, 4, 6, 6)  # 2 yığın, 4 baş, 6 token, 6 token

# Temel indeksleme
first_batch = x[0]              # Şekil: (4, 6, 6) - ilk yığın, tüm başlar
first_head = x[0, 0]            # Şekil: (6, 6) - ilk yığın, ilk baş
single_value = x[0, 0, 0, 0]    # Şekil: () - tek bir sayı

# İki nokta üst üste ile dilimleme
first_two_batches = x[:2]       # Şekil: (2, 4, 6, 6) - ilk 2 yığın
all_but_last_token = x[:, :, :-1, :]  # Şekil: (2, 4, 5, 6) - son token'ı kaldır
every_other_head = x[:, ::2]    # Şekil: (2, 2, 6, 6) - 0 ve 2 numaralı başlar

# Negatif indeksler sondan sayar
last_token = x[:, :, -1, :]     # Şekil: (2, 4, 6) - her dizideki son token

# Boolean maskeleme (koşula göre filtrele)
mask = torch.tensor([True, False, True, False])
filtered_heads = x[0, mask]     # Şekil: (2, 6, 6) - sadece 0 ve 2 numaralı başlar

print(f'Orijinal şekil: {x.shape}')
print(f'İlk yığın şekli: {first_batch.shape}')
print(f'Son token hariç tümü: {all_but_last_token.shape}')
print(f'Filtrelenmiş başlar: {filtered_heads.shape}')

## Dikkat: Transformer'ların Kalbi

**Büyük resim:** Dikkat, tüm kelimelerin ağırlıklı ortalamasını hesaplar, burada ağırlıklar ilgililik skorlarından gelir. "The cat sat on the mat" cümlesini okumak gibi, "sat"ı işlerken "cat"e (kim oturdu?) ve "mat"e (nereye oturdu?) geri bakarsınız.

Bunu 3 adımda oluşturacağız:
1. Temel matematik (4 işlem)
2. Nedensel maskeleme ekle (geleceğe bakmayı engelle)
3. Üretim kısayolu (PyTorch hepsini yapar)

In [ ]:
import torch
import torch.nn.functional as F

# ===== Dikkatin Temel Matematiği =====
# Dikkat: Sorgu (ne arıyorum?), Anahtar (ne var?), Değer (ne döndüreceğim?)

# 2 yığında 5 token için gömmeleri simüle et
batch, seq_len, d_head = 2, 5, 64
Q = torch.randn(batch, seq_len, d_head)  # (2, 5, 64)
K = torch.randn(batch, seq_len, d_head)
V = torch.randn(batch, seq_len, d_head)

# Adım 1: Skorları hesapla (her token diğerlerine ne kadar uyuyor?)
# @ operatörü matris çarpımıdır (torch.matmul ile aynı)
# K.transpose(-2, -1) son iki boyutu değiştirir: (2, 5, 64) → (2, 64, 5)
# Sonuç: (2, 5, 64) @ (2, 64, 5) → (2, 5, 5) — yığın başına 5×5 skor ızgarası
scores = Q @ K.transpose(-2, -1)
print(f'Skorlar şekli: {scores.shape}')  # (2, 5, 5)

# Adım 2: Ölçeklendir (softmax doygunluğunu engelle)
# Ölçeklendirme olmadan, büyük iç çarpımlar → softmax maksimum için ~1, diğerleri için ~0 verir
scores = scores / (d_head ** 0.5)  # sqrt(64) = 8'e böl

# Adım 3: Softmax (skorları olasılıklara dönüştür)
# dim=-1 "son boyut boyunca" demektir (anahtarlar boyunca)
attn_weights = torch.softmax(scores, dim=-1)  # her satır 1'e toplar

# Adım 4: Ağırlıklı toplam (olasılıkları kullanarak değerleri karıştır)
output = attn_weights @ V  # (2, 5, 64)

print(f'Çıktı şekli: {output.shape}')  # giriş ile aynı: (2, 5, 64)
print(f'Dikkat ağırlıkları 1'e toplar: {attn_weights[0, 0].sum():.4f}')
print(f'\nToken 2\'nin dikkat ağırlıkları: {attn_weights[0, 2]}')
print('(token 2\'nin 5 token\'ın her birine ne kadar dikkat ettiğini gösterir)')

## Token ve Konum Gömmeleri

Her LLM, token kimliklerini yoğun vektörlere dönüştürerek başlar.

**Problem:** Sinir ağları "cat" gibi ham metni işleyemez. Token kimlikleri ("cat" için `5` gibi) keyfidir - kimlik 5, 6'ya 500'den daha "yakın" değildir.

**Çözüm:** Her token kimliğini öğrenilmiş bir vektöre (GPT-2 için 768 sayı) eşle. Benzer kelimeler eğitim yoluyla benzer vektörler öğrenir.

In [ ]:
# ===== Token ve Konum Gömmeleri =====
# GPT-2 boyutları
vocab_size, d_model, max_seq_len = 50257, 768, 1024

# Token gömmeleri: bir arama tablosu (sözlük gibi: token_kimliği → vektör)
# nn.Embedding vocab_size satır ve d_model sütunlu bir tablo oluşturur
# Her satır bir token'ı temsil eden 768 boyutlu bir vektördür
token_embedding = nn.Embedding(vocab_size, d_model)

# Rastgele token kimlikleri oluştur (bunların tokenizer'dan geldiğini varsay)
token_ids = torch.randint(0, vocab_size, (2, 5))  # 2 cümle, her biri 5 token

# Gömmeleri ara (sadece tabloya indeksleme!)
token_vectors = token_embedding(token_ids)
print(f'Token gömmeleri: {token_vectors.shape}')  # (2, 5, 768)

# Konum gömmeleri: dizide nerede (token 0, token 1, vb.)
# Aynı fikir: i satırının "konum i"yi temsil ettiği bir arama tablosu
pos_embedding = nn.Embedding(max_seq_len, d_model)
position_ids = torch.arange(5).unsqueeze(0).expand(2, -1)  # [[0,1,2,3,4], [0,1,2,3,4]]
pos_vectors = pos_embedding(position_ids)
print(f'Konum gömmeleri: {pos_vectors.shape}')  # (2, 5, 768)

# Birleştir: eleman bazında toplama (aynı konum, aynı şekil!)
# Neden birleştirmek yerine ekle? Toplama boyutu 768'de tutar (1536 değil)
# Model HEM anlam HEM de konumu aynı vektörde kodlamayı öğrenir
input_embeddings = token_vectors + pos_vectors
print(f'Birleştirilmiş: {input_embeddings.shape}')  # (2, 5, 768)

# Parametre sayımı (kaç sayı öğreneceğiz?)
token_params = vocab_size * d_model   # 50.257 token × 768 boyut
pos_params = max_seq_len * d_model    # 1.024 konum × 768 boyut
print(f'Token parametreleri: {token_params:,}')      # 38.597.376
print(f'Konum parametreleri: {pos_params:,}')        # 786.432
print(f'Toplam: {token_params + pos_params:,}')      # 39.383.808

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Oyuncak sınıflandırma için basit 2 katmanlı MLP (kendi içinde bağımsız)
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)

# 5 Adımlı Eğitim Tarifi:
# 1. İleri geçiş -> tahminleri al
# 2. Kaybı hesapla -> hatayı ölç
# 3. Geri geçiş -> gradyanları hesapla
# 4. Gradyanları kırp -> patlamaları engelle
# 5. Ağırlıkları güncelle -> parametreleri ayarla

# Kurulum: model, optimizatör, kayıp fonksiyonu, sahte veri
model = SimpleMLP(16, 32, 2)  # girdi 16-boyut, gizli 32-boyut, çıktı 2 sınıf
optimizer = optim.Adam(model.parameters(), lr=1e-3)  # uyarlanabilir öğrenme oranı
loss_fn = nn.CrossEntropyLoss()  # sınıflandırma için

# Sahte eğitim verisi (batch_size=64)
inputs = torch.randn(64, 16)          # 64 örnek, her biri 16 özellik
labels = torch.randint(0, 2, (64,))   # 64 etiket (sınıf 0 veya 1)

# Eğitim döngüsü (3 dönem)
for epoch in range(3):
    # ===== Eğitim Aşaması =====
    model.train()  # Dropout/batch norm'u etkinleştir (varsa)

    # Adım 1: Eski gradyanları sıfırla (varsayılan olarak birikiyorlar!)
    optimizer.zero_grad(set_to_none=True)  # set_to_none bellek tasarrufu sağlar

    # Adım 2: İleri geçiş
    logits = model(inputs)  # Tahminleri al (ham skorlar)

    # Adım 3: Kaybı hesapla
    loss = loss_fn(logits, labels)  # Ne kadar yanlışız?

    # Adım 4: Geri geçiş (gradyanları hesapla)
    loss.backward()  # Her parametre için .grad'ı doldur

    # Adım 5: Gradyan kırpma (patlayan gradyanları engeller)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    # Adım 6: Ağırlıkları güncelle
    optimizer.step()  # Gradyanları kullanarak parametreleri ayarla

    print(f"Dönem {epoch}: kayıp={loss.item():.4f}")

# ===== Değerlendirme Aşaması =====
model.eval()  # Dropout/batch norm'u devre dışı bırak
with torch.no_grad():  # Gradyanları izleme (bellek tasarrufu)
    preds = model(inputs).argmax(dim=-1)  # Sınıf tahminlerini al
    accuracy = (preds == labels).float().mean()  # Doğru olanların oranı
    print(f"Doğruluk: {accuracy:.2f}")

print()
print("✅ Ana noktalar:")
print("  - .backward() her parametrenin .grad'ını doldurur")
print("  - Optimizatör ağırlıkları ayarlamak için gradyanları kullanır")
print("  - Gradyan kırpma kayıp sıçramalarını engeller (LLM'ler için kritik!)")
print("  - .eval() ve no_grad() değerlendirme sırasında bellek tasarrufu sağlar")

In [ ]:
# Tam gömme modülü
class GPT2Embeddings(nn.Module):
    def __init__(self, vocab_size, max_seq_len, d_model, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, token_ids):
        batch_size, seq_len = token_ids.shape
        position_ids = torch.arange(seq_len, device=token_ids.device)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, -1)
        
        token_emb = self.token_embedding(token_ids)
        pos_emb = self.pos_embedding(position_ids)
        
        embeddings = token_emb + pos_emb
        embeddings = self.dropout(embeddings)
        return embeddings

# Test et
embed_layer = GPT2Embeddings(50257, 1024, 768)
token_ids = torch.randint(0, 50257, (2, 10))
output = embed_layer(token_ids)
print(f"Gömme çıktısı: {output.shape}")  # (2, 10, 768)

# Toplam parametreler
total_params = sum(p.numel() for p in embed_layer.parameters())
print(f"Toplam gömme parametreleri: {total_params:,}")  # 39.383.808

## Eğitim Döngüsü

**Ana kavramlar:**
- `nn.Module`: Sinir ağı katmanları için temel sınıf. Modeliniz bundan türer.
- `super().__init__()`: Ebeveyn sınıfın başlatılmasını çağırır (gerekli şablon kod)
- `nn.Linear(in, out)`: Bir matris çarpım katmanı (in×out ağırlık matrisi)
- `nn.ReLU()`: Aktivasyon fonksiyonu — pozitif değerleri tutar, negatif olanları sıfırlar
- `CrossEntropyLoss`: Sınıflandırma tahminlerinin ne kadar yanlış olduğunu ölçer
- `optimizer.zero_grad()`: Eski gradyanları temizler (varsayılan olarak birikiyorlar!)
- `.backward()`: Otomatik türev alma yoluyla gradyanları hesaplar
- `.step()`: Hesaplanan gradyanları kullanarak ağırlıkları günceller

In [ ]:
# Adım 2: Nedensel Maskeleme Ekle (Hile Yapmayı Engelle)

# Problem: Token 2, token 3 ve 4'ü görebilirse eğitim sırasında hile yapabilir!
# Çözüm: Skorlarını -inf'e ayarlayarak gelecek konumları blokla

# Maskenin görünümü (False = izin ver, True = blokla):
# Token 0 görebilir: [0]           ← sadece kendisi
# Token 1 görebilir: [0, 1]        ← geçmiş + kendisi
# Token 2 görebilir: [0, 1, 2]     ← geçmiş + kendisi
# Token 3 görebilir: [0, 1, 2, 3]
# Token 4 görebilir: [0, 1, 2, 3, 4]

seq_len = 5
causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()

# Softmax'tan önce maske uygula
scores = Q @ K.transpose(-2, -1) / (d_head ** 0.5)
scores = scores.masked_fill(causal_mask, float('-inf'))  # -inf softmax'tan sonra 0 olur
attn_weights = torch.softmax(scores, dim=-1)

print("Nedensel dikkat ağırlıkları (token 2 sadece token 0,1,2'yi görebilir):")
print(attn_weights[0, 2])  # pozisyon 3 ve 4 sıfır
print("\nArtık model önce 'mat'ı görmeden 'mat'ı tahmin etmeyi öğrenir!")

In [ ]:
# Adım 3: Üretim Kısayolu (Tek Satır)

# Neler olduğunu anlamak için 4 adımlı manuel süreci öğrendiniz.
# Pratikte PyTorch hepsini sizin için yapar:

output = F.scaled_dot_product_attention(
    Q, K, V,
    is_causal=True  # otomatik olarak nedensel maskeleme uygular
)

print(f"Çıktı şekli: {output.shape}")  # (2, 5, 64)

# Manuel yerine bunu neden kullanalım?
# - 2-4× daha hızlı (FlashAttention kullanır)
# - Daha az bellek (tam dikkat matrisini saklamaz)
# - Uç durumları işler (sayısal kararlılık, dropout, maske yayma)

print("\n✅ Manuel vs. üretim ne zaman kullanılır:")
print("   Öğrenme: Matematiği anlamak için manuel yazın")
print("   Üretim: Her zaman F.scaled_dot_product_attention kullanın")

In [ ]:
seq_len = 5
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
print(mask)

## Yayma örnekleri
Önyargı ekleme ve maskeleme yayması.

In [ ]:
x = torch.randn(3, 4)
y = x + 5
batch = torch.randn(32, 10, 768)
bias = torch.randn(768)
result = batch + bias  # önyargı yayılır
scores = torch.randn(4, 8, 10, 10)
mask = torch.triu(torch.ones(1, 1, 10, 10), 1)
masked = scores + mask * -1e9
print('sonuç şekli:', result.shape)

## Temel İşlemler: Eleman Bazında, Matmul, İndirgeme

**Eleman bazında işlemler** konum konum çalışır - iki elektronik tabloyu hücre hücre toplamak gibi. `a` ve `b`'nin her ikisi de 3×4 tensör ise, o zaman `a + b`, `a[0,0]`'ı `b[0,0]`'a, `a[0,1]`'i `b[0,1]`'e ekler ve böyle devam eder. Aynı şekil girer, aynı şekil çıkar.

In [ ]:
a = torch.randn(3, 4)
b = torch.randn(3, 4)
add = a + b
mul = a * b
square = a ** 2
exp = torch.exp(a)
x = torch.randn(32, 10, 64)
W = torch.randn(64, 128)
y = x @ W
total = a.sum()
row_sums = a.sum(dim=1, keepdim=True)
col_means = a.mean(dim=0)
max_vals, max_idx = a.max(dim=1)
c = torch.cat([a, b], dim=0)
d = torch.stack([a, b], dim=0)
logits = torch.randn(3, 5)
probs = torch.softmax(logits, dim=-1)
print('y şekli:', y.shape)
print('probs satır toplamları:', probs.sum(dim=-1))

## Autograd: Otomatik Gradyanlar

**Gradyan nedir?** Türev - girdi değiştiğinde çıktı ne kadar değişir?

**Neden önemli?** Bir sinir ağını eğitmek milyonlarca parametreyi ayarlamak demektir. Gradyanlar bize hangi yönde ayarlama yapacağımızı söyler. PyTorch'un autograd'ı bunu otomatik yapar.

In [ ]:
# Basit hesaplama grafiği: x → y → z
x = torch.tensor([2.0, 3.0], requires_grad=True)  # x üzerindeki işlemleri izle
y = x ** 2        # y = [4.0, 9.0]
z = y.sum()       # z = 13.0

# Gradyanları otomatik olarak hesapla
z.backward()      # "x'i değiştirirsem z nasıl değişir?"
print('Gradyanlar:', x.grad)  # tensor([4., 6.])

# Ne oldu?
# z = (x²).sum() → dz/dx = 2x
# x=[2, 3]'te gradyanlar 2*[2, 3] = [4, 6]
# backward() bunu grafikte geriye yürüyerek hesapladı!

print('\n✅ Manuel kontrol: dz/dx = 2x')
print(f'   x=[2, 3]'te: 2*x = {2 * x.detach()}')

# İzlemeyi ne zaman durduralım (çıkarım sırasında bellek tasarrufu):
print('\n--- Gradyan izlemeyi durdurma ---')

# Seçenek 1: Bağlam yöneticisi (bir kod bloğu için)
with torch.no_grad():
    y_no_grad = x * 2  # gradyan izleme yok
    print(f'Gradyan hesaplanmadı: {y_no_grad}')

# Seçenek 2: Ayır (tek bir tensör için)
detached = x.detach()  # yeni tensör, gradyan geçmişi yok
print(f'Ayrılmış tensör: {detached}')

## Özet

LLM geliştirme için tüm temel PyTorch işlemlerini gördünüz:

✅ **Tensör temelleri** - oluşturma, dtype'lar, cihazlar
✅ **Tekrarlanabilirlik** - hata ayıklama için tohum
✅ **Boyut oluşturma** - 1D → 2D → 3D → 4D ilerlemesi
✅ **Yeniden şekillendirme & indeksleme** - çok boyutlu veride gezinme
✅ **Dikkat mekanizması** - manuel + üretim kalıpları
✅ **Nedensel maskeleme** - gelecek token'a bakmayı engelleme
✅ **Gömmeler** - token + konum temsilleri
✅ **Yayma** - otomatik şekil genişletme
✅ **Matematik işlemleri** - eleman bazında, matmul, indirgeme
✅ **Autograd** - otomatik türev alma
✅ **Eğitim döngüleri** - gradyan kırpma ile ileri, geri, optimize et

**Sonraki adımlar:** Bölüm 7 size gerçek metin verisini eğitim için nasıl hazırlayacağınızı gösterecek!